# Cloud TTS — Colab worker

Runs **Chatterbox** (natural voice cloning) and **Kokoro** (fast fallback) on a free Colab GPU and connects them to your Render gateway.

### Before you start

1. Deploy this repo to Render (Blueprint `render.yaml`) and copy `WORKER_TOKEN`.
2. Add two Colab **Secrets** (key icon in the left sidebar):
   - `GATEWAY_URL` → `https://your-service.onrender.com`
   - `WORKER_TOKEN` → the worker token from Render
3. Set `REPO_URL` below to your GitHub repo.
4. Optional but recommended: put `male.wav` and `female.wav` (10–15 s each) in `MyDrive/cloud_tts_voices/`.

Then **Runtime → Change runtime type → T4 GPU**, and run all cells.


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB VRAM")
else:
    print("No GPU. Kokoro still works; Chatterbox will be slow (CPU).")


CUDA available: True
Tesla T4
14.6 GB VRAM


## 1. Install dependencies

Takes a few minutes the first time. If Colab asks to **restart the runtime** after installing, do it and continue from the next cell.


In [4]:
%%bash
set -e

echo "=== Install Python 3.12 ==="
apt-get update -qq
apt-get install -y python3.12 python3.12-venv python3.12-dev espeak-ng ffmpeg

echo "=== Create isolated environment ==="
python3.12 -m venv /content/tts-env

PY=/content/tts-env/bin/python

echo "=== Install pip and dependencies ==="
$PY -m pip install -q --upgrade pip setuptools wheel

echo "=== Install PyTorch with CUDA ==="
$PY -m pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu126

echo "=== Install Chatterbox and Kokoro ==="
$PY -m pip install "chatterbox-tts==0.1.7" "kokoro==0.7.16" soundfile requests

echo "=== Verify ==="
$PY - <<'PY'
import sys
import torch
import chatterbox
import kokoro
import soundfile

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Chatterbox:", chatterbox.__file__)
print("Kokoro:", kokoro.__file__)
print("All imports successful")
PY

=== Install Python 3.12 ===
Reading package lists...
Building dependency tree...
Reading state information...
python3.12 is already the newest version (3.12.3-1ubuntu0.17).
python3.12 set to manually installed.
espeak-ng is already the newest version (1.51+dfsg-12build1).
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
The following additional packages will be installed:
  python3-pip-whl python3-setuptools-whl
The following NEW packages will be installed:
  python3-pip-whl python3-setuptools-whl python3.12-dev python3.12-venv
0 upgraded, 4 newly installed, 0 to remove and 79 not upgraded.
Need to get 2,927 kB of archives.
After this operation, 3,291 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 python3-pip-whl all 24.0+dfsg-1ubuntu1.3 [1,707 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 python3-setuptools-whl all 68.1.2-2ubuntu1.2 [716 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-upd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Get the worker code

In [5]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/pravesh34/cloud-tts.git"   # your repo
WORKDIR = pathlib.Path("/content/cloud_tts")

if WORKDIR.exists():
    print("Using existing checkout:", WORKDIR)
else:
    if "CHANGE_ME" in REPO_URL:
        raise SystemExit("Set REPO_URL to your GitHub repo (the one you deployed to Render).")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)], check=True)

os.chdir(WORKDIR)
print("working directory:", pathlib.Path.cwd())


working directory: /content/cloud_tts


## 3. Gateway URL and worker token

Read from Colab Secrets. If a secret is missing you will be prompted.


In [7]:
import os
from getpass import getpass

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

gateway = os.environ.get("GATEWAY_URL") or secret("GATEWAY_URL")
if not gateway:
    gateway = input("Gateway URL: ").strip()

token = os.environ.get("WORKER_TOKEN") or secret("WORKER_TOKEN")
if not token:
    token = getpass("Worker token (hidden): ")

os.environ["GATEWAY_URL"] = gateway.rstrip("/")
os.environ["WORKER_TOKEN"] = token
print("gateway:", os.environ["GATEWAY_URL"])
print("token set:", bool(os.environ["WORKER_TOKEN"]))


gateway: https://cloud-tts-gateway.onrender.com
token set: True


## 4. Reference voices (optional)

Chatterbox clones the voice in `worker/voices/male.wav` and `female.wav`.

- Use your own recordings or public-domain audio you have rights to.
- 10–15 seconds, one speaker, clean, 24 kHz+, no music.
- Without these files the worker uses Chatterbox's built-in voice, and Kokoro voices still work.


In [8]:
import shutil
from pathlib import Path

VOICE_DIR = WORKDIR / "worker" / "voices"
DRIVE_VOICES = Path("/content/drive/MyDrive/cloud_tts_voices")
MOUNTED = False

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    MOUNTED = True
except Exception as exc:
    print("Drive not mounted:", exc)

if MOUNTED and DRIVE_VOICES.exists():
    for name in ("male.wav", "female.wav"):
        src = DRIVE_VOICES / name
        if src.exists():
            shutil.copy2(src, VOICE_DIR / name)
            print("copied", name)
        else:
            print("missing (optional):", src)
else:
    print("No Drive voice folder. Use the next cell to upload clips, or skip.")


Mounted at /content/drive
No Drive voice folder. Use the next cell to upload clips, or skip.


### Optional: upload clips now

Set `UPLOAD_VOICES = True`, run this cell, and pick your WAV files. They are copied into `worker/voices/` for this session. Put them in Drive to reuse them next time.


In [9]:
UPLOAD_VOICES = False

if UPLOAD_VOICES:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.lower().endswith(".wav"):
            (VOICE_DIR / name).write_bytes(data)
            print("saved", name)


## 5. Cache models in Drive (recommended)

The first Chatterbox run downloads ~3 GB. Caching in Drive makes later sessions much faster. This cell is harmless if Drive is not mounted.


In [10]:
import os
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
if drive_root.exists():
    cache = drive_root / "cloud_tts_cache"
    cache.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache)
    os.environ["TORCH_HOME"] = str(cache / "torch")
    print("model cache:", cache)
else:
    print("Using ephemeral cache (models re-download each session).")


model cache: /content/drive/MyDrive/cloud_tts_cache


In [17]:
import subprocess, sys
print(subprocess.run(["/content/tts-env/bin/python", "-c", """
import sys
sys.path.insert(0, "/content/cloud_tts/worker")
from engines.chatterbox import ChatterboxEngine
e = ChatterboxEngine(voices_dir="/content/cloud_tts/worker/voices")
e.load()
print("ready:", e.sample_rate)
"""], capture_output=True, text=True).stdout)

[chatterbox] loading on cuda (first run downloads ~3 GB)



## 6. Run the worker

This cell blocks and streams the worker log. Keep the Colab tab open.

- The worker sends a heartbeat every 15 s, long-polls the gateway, and loads a model only when asked.
- First load is slow (download + init). Later loads are much faster.
- Stop this cell (or close the tab) to disconnect. **Colab cannot be restarted remotely** — re-run the notebook instead.


In [ ]:
import subprocess, sys, os

env = {**os.environ}
VENV_PY = "/content/tts-env/bin/python"
PY = VENV_PY if os.path.exists(VENV_PY) else sys.executable
print(f"starting worker with {PY}", flush=True)
proc = subprocess.Popen([PY, "worker.py"], cwd=str(WORKDIR / "worker"), env=env)
print("worker pid:", proc.pid)
proc.wait()
print("worker stopped — re-run this cell to reconnect.")


starting worker with /content/tts-env/bin/python
worker pid: 24185


## Notes

- **Render free tier** spins down after 15 minutes of no traffic. While the worker polls, the service stays awake. Stop the runtime when you are done so you do not burn your 750 monthly instance hours.
- **Jobs are ephemeral.** Audio is stored in Render's `/tmp` SQLite and deleted after ~45 minutes. Download what you need.
- **Chatterbox is watermarked** with Resemble's inaudible Perth mark.
- **Licensing:** Chatterbox is MIT, Kokoro is Apache-2.0. Only clone voices you have permission to use.
- If you hit CUDA OOM: switch the voice to a Kokoro profile, or restart the runtime.


In [ ]:
import sys
print("Notebook Python:", sys.executable)

! /content/tts-env/bin/python -c "import torch, chatterbox, kokoro; print('Imports OK'); print('CUDA:', torch.cuda.is_available())"